# Official ColBERTv2 baseline on BEIR (Colab)

Do pipeline **goc** cua ColBERTv2 voi **dung settings cua ban mini**
(`query_maxlen=32`, `doc_maxlen=180`, `nbits=2`, top-100) de so sanh
`Official` vs `Mini Base` vs `CAD` tren cung mot dataset cho cong bang.

Ket qua ghi vao `outputs/<dataset>/official_baseline.json`; `run_evaluate.py`
se tu doc file nay va hien hang `Official ColBERTv2`.

**Dieu kien:** chay `Mini_ColBERTv2_MultiBEIR.ipynb` truoc de co san
`outputs/<dataset>/`. Can Colab GPU (T4).

> Luu y: so trong paper ColBERTv2 dung `doc_maxlen=300` nen KHONG so truc tiep
> duoc voi ban mini. Notebook nay chay lai official voi `180` cho dong dieu kien.

In [ ]:
# 1. Clone the official ColBERT repo (GPU required for indexing)
!nvidia-smi
!git clone https://github.com/stanford-futuredata/ColBERT.git || true
%cd ColBERT
!pip -q install -e .
!pip -q install faiss-gpu-cu12

In [ ]:
# 2. Pin transformers<5 (colbert-ai compatibility) + helpers
!pip install -q "transformers<5" jedi ninja
import torch, transformers
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)

In [ ]:
# 3. Clone the mini repo (for outputs/ + baseline scripts) and copy scripts into ColBERT
%cd /content
!git clone https://github.com/HoangNguyen689/colbert-hoangnd.git || true
!git -C colbert-hoangnd pull
MINI_DIR = "/content/colbert-hoangnd"
!cp {MINI_DIR}/baselines/official/prepare_beir.py /content/ColBERT/
!cp {MINI_DIR}/baselines/official/run_official_colbertv2.py /content/ColBERT/
!cp {MINI_DIR}/baselines/official/evaluate_official.py /content/ColBERT/
!ls /content/ColBERT/*.py

In [ ]:
# 4. Run official prepare -> index+search -> evaluate for each dataset
#    Settings are pinned to match the mini pipeline (query/doc maxlen, nbits=2, k=100).
import os
from pathlib import Path

DATASETS = ['nfcorpus', 'scidocs', 'fiqa', 'arguana']
MINI_DIR = Path("/content/colbert-hoangnd")

os.chdir("/content/ColBERT")
for ds in DATASETS:
    print(f'\n==================== {ds} ====================')
    out_dir = MINI_DIR / 'outputs' / ds
    if not out_dir.exists():
        print(f'[skip] {out_dir} not found - run Mini_ColBERTv2_MultiBEIR.ipynb first')
        continue
    !python prepare_beir.py --dataset {ds} --out /content/ColBERT/data
    !python run_official_colbertv2.py --dataset {ds} --data /content/ColBERT/data/{ds}_colbert --nbits 2 --k 100 --doc-maxlen 180 --query-maxlen 32
    !python evaluate_official.py --dataset {ds} --json-out {out_dir}/official_baseline.json

In [ ]:
# 5. Print per-dataset comparison tables (Official vs Mini Base vs IDF/Adaptive/CAD)
import os
os.chdir("/content/colbert-hoangnd")
DATASETS = ['nfcorpus', 'scidocs', 'fiqa', 'arguana']
for ds in DATASETS:
    out = f'outputs/{ds}'
    print(f'\n==================== {ds} ====================')
    !python run_evaluate.py --dataset {ds} --output_dir {out} --qrels {out}/qrels_test.json